In [146]:
import numpy as np
import pandas as pd
import os
import re
from numpy.linalg import norm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD
import plotly.express as px
import random

In [147]:
random.seed(101)

In [148]:
LIB=pd.read_csv("Corpus_LIB.csv")

In [149]:
LIB

,book_title,author,book_len,n_chaps,avg_sent_len
0,The Old Curiosity Shop,Charles Dickens,218729,73,17.999424
1,The Mystery of Edwin Drood,Charles Dickens,96436,23,12.012456
2,Dombey and Son,Charles Dickens,356587,62,15.899189
3,Nicholas Nickleby,Charles Dickens,324259,65,14.234997
4,Bleak House,Charles Dickens,354643,67,13.623872
5,Martin Chuzzlewit,Charles Dickens,338838,55,13.576872
6,David Copperfield,Charles Dickens,357805,64,13.334513
7,A Tale of Two Cities 1,Charles Dickens,17520,6,15.077453
8,A Tale of Two Cities 2,Charles Dickens,70973,24,13.998619
9,A Tale of Two Cities 3,Charles Dickens,48583,15,13.394817


In [150]:
LIB["label"]=LIB["book_title"]

In [151]:
TFIDF=pd.read_csv("tfidf_L2.csv",index_col=0)

In [152]:
TFIDF

,abel,ada,affery,agnes,allan,allen,amy,antoine,arthur,ath,...,willet,windlass,winkle,wold,woodcourt,woodman,wopsle,wrayburn,wren,yo
book_title,,,,,,,,,,,,,,,,,,,,,
A Tale of Two Cities 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.055247,0.000000,0.00000,0.000000,0.000000
A Tale of Two Cities 2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.077648,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
A Tale of Two Cities 3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.058555,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Barnaby Rudge,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.218013,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Bleak House,0.000000,0.307637,0.000000,0.000000,0.069385,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.065768,0.119544,0.000000,0.000000,0.00000,0.000000,0.000000
David Copperfield,0.000000,0.000000,0.000000,0.219673,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Dombey and Son,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
Great Expectations,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.145311,0.00000,0.000000,0.000000
Hard Times 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000


In [153]:
pca_engine = PCA(n_components=5)
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF), index=TFIDF.index)
DCM.columns = ['PC{}'.format(i) for i in DCM.columns]
DCM

,PC0,PC1,PC2,PC3,PC4
book_title,,,,,
A Tale of Two Cities 1,-0.237249,-0.515913,-0.393326,0.206946,0.000601
A Tale of Two Cities 2,-0.260106,-0.580502,-0.451840,0.249186,0.000861
A Tale of Two Cities 3,-0.251600,-0.556875,-0.430726,0.234413,0.000780
Barnaby Rudge,-0.083428,-0.028103,0.095299,-0.252547,-0.204023
Bleak House,-0.080208,-0.027398,0.092623,-0.241981,-0.051634
David Copperfield,-0.082590,-0.027704,0.093566,-0.244858,-0.052424
Dombey and Son,-0.082625,-0.027721,0.093639,-0.245179,-0.054888
Great Expectations,-0.082599,-0.027709,0.093595,-0.245052,-0.118989
Hard Times 1,-0.316809,0.661484,-0.283222,0.201124,0.000743


In [154]:
COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = TFIDF.columns
COMPS.index.name = 'term_str'
COMPS.T

term_str,abel,ada,affery,agnes,allan,allen,amy,antoine,arthur,ath,...,willet,windlass,winkle,wold,woodcourt,woodman,wopsle,wrayburn,wren,yo
PC0,-0.000592,-0.003188,-0.003761,-0.002344,-0.000719,-0.000768,-0.005197,-0.004513,-0.007809,-0.002585,...,-0.002350,-0.002187,-0.001944,-0.000682,-0.001239,-0.001694,-0.001551,0.020194,0.031186,-0.002455
PC1,-0.000226,-0.001211,-0.002270,-0.000874,-0.000273,-0.000286,-0.003136,-0.011159,-0.004713,0.006024,...,-0.000880,0.005097,-0.000725,-0.000259,-0.000470,-0.004094,-0.000578,0.001464,0.002285,0.005570
PC2,0.000817,0.004375,0.020238,0.003156,0.000987,0.001034,0.027965,-0.009259,0.042021,-0.002764,...,0.003190,-0.002339,0.002618,0.000935,0.001700,-0.003336,0.002088,-0.002960,-0.004660,-0.002502
PC3,-0.002415,-0.012817,0.016577,-0.009261,-0.002891,-0.003037,0.022905,0.005695,0.034419,0.002212,...,-0.009480,0.001872,-0.007689,-0.002740,-0.004981,0.001969,-0.006131,0.003620,0.005815,0.001906
PC4,-0.000856,-0.003427,0.000042,-0.002484,-0.000773,-0.000865,0.000058,0.000024,0.000088,0.000010,...,-0.009595,0.000009,-0.002191,-0.000733,-0.001332,0.000007,-0.003730,0.000011,0.000030,0.000007


In [155]:
def vis_pcs(DCM, a, b, label='book_title', hover_name='label'):
    return px.scatter(DCM, f"PC{a}", f"PC{b}", 
                    color=LIB[label], 
                    hover_name=LIB[hover_name], 
                    size=LIB['book_len'],
                    marginal_x='box', height=800)

In [ ]:
def vis_loadings(COMPS, a=0, b=1, hover_name='term_str'):
    return px.scatter(COMPS.reset_index(), f"PC{a}", f"PC{b}",
                      text='term_str', 
                      marginal_x='box', 
                      height=800)

In [157]:
vis_pcs(DCM, 0, 1)

In [158]:
vis_loadings(COMPS, 0, 1)

In [159]:
vis_pcs(DCM, 2,3)

In [160]:
vis_loadings(COMPS, 2, 3)

In [161]:
top_terms_sk = {}
data = []
for i in range(5):
    for j in [0, 1]:
        data.append((i, j, ' '.join(COMPS.sort_values(f'PC{i}', ascending=bool(j)).head(10).index.to_list())))
comp_strs = pd.DataFrame(data)
comp_strs.columns =  ['pc', 'pole', 'gloss']
comp_strs = comp_strs.set_index(['pc', 'pole'])
comp_strs.unstack()

gloss  \
pole                                                  0   
pc                                                        
0     boffin wegg bella fledgeby eugene lammle rider...   
1     bounderby gradgrind sparsit louisa sissy steph...   
2     dorrit clennam pancks merdle meagles arthur fl...   
3     dorrit clennam lorry bounderby defarge pancks ...   
4     nicholas nickleby squeers ralph kate newman ke...   

                                                         
pole                                                  1  
pc                                                       
0     bounderby lorry gradgrind defarge sparsit loui...  
1     lorry defarge manette pross carton darnay luci...  
2     lorry defarge bounderby gradgrind manette pros...  
3     pecksniff pickwick dombey nicholas quilp olive...  
4     pecksniff hugh barnaby oliver haredale locksmi...

In [162]:
PCA_Components = pd.DataFrame(comp_strs.unstack())

In [163]:
DCM.to_csv("PCA_DCM.csv")
COMPS.to_csv("PCA_COMPS.csv")
PCA_Components.to_csv("PCA_Components.csv")